# Dataset pipeline analysis

Inspect saved diagnosis, cleaning, deduplication, and experiment artifacts.
The analysis cells load existing results; command cells print shell commands
without running them. Training tokenization at the end is explicitly opt-in.

Install the project requirements with `pip install -r src/requirements.txt`.
Open this notebook from the project root or the `notebooks` directory.


## Select a saved run

Change only `output_path` in the next cell to select another run. The directory
must contain the pipeline's saved `pipeline_config.json`. Source, text-column,
split, normalization, and tokenizer details come from that configuration.

For a first run, use `scripts/run_dataset_pipeline.py --config /absolute/path/to/settings.json
--output_path /absolute/path/to/run` from the project root. Standalone examples
below use the selected run's saved settings and absolute artifact paths.

Existing complete or partial outputs require `--overwrite` to replace them;
there is no automatic resume. Add `--save_dataset` to a sweep to export retained
rows, otherwise trials share their saved input and signatures. For a new
parameter configuration, copy the saved JSON to a separate file and choose a
separate trial destination; do not edit the run's provenance just to plot it.


In [ ]:
import json
import logging
import shlex
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_from_disk
from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / 'src' / 'analysis_results.py').is_file():
    project_root = project_root.parent
if not (project_root / 'src' / 'analysis_results.py').is_file():
    raise FileNotFoundError('Open the notebook from the project root or notebooks directory')
sys.path.insert(0, str(project_root / 'src'))

import analysis_results as analysis
import diagnostics as diagnostic

logging.basicConfig(level=logging.INFO)
_LOGGER = logging.getLogger(__name__)

# Change this one absolute path to analyze another run.
output_path = Path('/absolute/path/to/run')
config_path = output_path / 'pipeline_config.json'
pipeline_config = json.loads(config_path.read_text(encoding='utf-8'))
source_settings = pipeline_config['source']
text_column = source_settings['text_column']
configured_splits = list(source_settings['splits'])

diagnosis_path = output_path / 'diagnosis'
normalized_path = output_path / 'normalized'
exact_path = output_path / 'exact_dedup'
minhash_path = output_path / 'minhash_dedup'
experiments_path = output_path / 'experiments'
normalized_signatures_path = experiments_path / 'signatures_normalized'
exact_signatures_path = experiments_path / 'signatures_exact'
ngram_result_dirs = {
    'Before exact deduplication': experiments_path / 'ngram_normalized',
    'After exact deduplication': experiments_path / 'ngram_exact',
}
threshold_results_dir = experiments_path / 'threshold'
single_threshold_dir = experiments_path / 'threshold_single'
signature_ngram_sizes = list(dict.fromkeys([
    *pipeline_config['ngram_experiment']['ngram_sizes'],
    pipeline_config['threshold_experiment']['ngram_size'],
]))


def show_command(script, *arguments):
    command = [sys.executable, str(project_root / 'scripts' / script),
               '--config', str(config_path), *map(str, arguments)]
    display(Markdown('```bash\n' + shlex.join(command) + '\n```'))


display(pd.Series({
    'dataset': source_settings.get('local_path') or source_settings.get('dataset_id'),
    'text_column': text_column,
    'splits': source_settings['splits'],
    'normalization': pipeline_config['normalization'],
    'tokenizer': pipeline_config['tokenizer'],
}, name='Saved run settings'))


In [ ]:
display(Markdown('**Full pipeline.** Runs diagnosis, preparation, exact deduplication, '
                 'and selected MinHash deduplication. Add `--overwrite` to replace this run.'))
show_command('run_dataset_pipeline.py', '--output_path', output_path)
display(Markdown('**Standalone preparation.** Writes normalized datasets and cleaning metrics. '
                 'This loads the configured original source, so its files or Hub access must be available.'))
show_command('run_dataset_preparation.py', '--output_path', normalized_path)


In [ ]:
normalized_datasets = {}
for split in configured_splits:
    path = normalized_path / f'normalized_{split}'
    if path.is_dir():
        normalized_datasets[split] = load_from_disk(str(path))
    else:
        print(f'Missing optional normalized split: {path}. Run preparation above.')


In [ ]:
if 'train' in normalized_datasets:
    training_documents = normalized_datasets['train']
    diagnostic.display_dataset_information(training_documents)
    display(training_documents.select(range(min(10, len(training_documents)))).to_pandas())


## Dataset diagnosis

Diagnosis describes the original mapped training split using the configured
tokenizer: document counts, full token lengths without truncation, token
frequencies, Zipf plots, and word clouds. These are raw-input statistics, before
normalization and deduplication.

The saved token-frequency CSV contains token IDs and text. The existing plots
can use these labels directly, so analysis does not need to download a tokenizer
or tokenize the source again.


In [ ]:
display(Markdown('**Standalone diagnosis.** Uses the source and tokenizer in the saved configuration.'))
show_command('run_dataset_diagnosis.py', '--output_path', diagnosis_path)


In [ ]:
document_lengths = token_frequencies = diagnosis_summary = None
tokenizer = None
diagnosis_files = ['document_lengths.csv', 'token_frequencies.csv', 'diagnosis_summary.json']
if all((diagnosis_path / name).is_file() for name in diagnosis_files):
    document_lengths, token_frequencies, diagnosis_summary = analysis.load_diagnosis(diagnosis_path)
    token_labels = dict(zip(token_frequencies['token_id'], token_frequencies['token_text']))
    # The existing plotting helper only needs this token-ID lookup.
    tokenizer = SimpleNamespace(convert_ids_to_tokens=token_labels.__getitem__)
else:
    print('Diagnosis artifacts are missing. Run the command above to generate them.')


In [ ]:
if token_frequencies is not None:
    _LOGGER.info('Saved vocabulary size: %d', len(token_frequencies))
    _LOGGER.info('Loaded diagnosis for %d documents', diagnosis_summary['document_count'])


In [ ]:
id_dist = {}
sequence_lengths = []
if token_frequencies is not None:
    id_dist = dict(zip(token_frequencies['token_id'], token_frequencies['count']))
    sequence_lengths = document_lengths['token_count'].to_numpy()


In [ ]:
if token_frequencies is not None:
    display(token_frequencies.nlargest(10, 'count'))


In [ ]:
if diagnosis_summary is not None:
    display(pd.Series(diagnosis_summary, name='Raw training diagnosis'))


In [ ]:
if diagnosis_summary is not None and sum(id_dist.values()) > 0:
    fig, axes = diagnostic.plot_dataset_diagnosis(
        sequence_lengths, id_dist, diagnosis_summary, tokenizer,
    )
    plt.show()


In [ ]:
if token_frequencies is not None:
    display(Markdown('**Highest-ranked token labels saved by diagnosis**'))
    display(token_frequencies.nlargest(10, 'count'))
    display(Markdown('**Lowest-ranked observed token labels**'))
    observed_tokens = token_frequencies.loc[token_frequencies['count'] > 0]
    display(observed_tokens.nsmallest(10, 'count'))


### Interpret the selected run

- Inspect frequent tokens for boilerplate, markup, or repeated templates. Common
  words and punctuation can be valid language-model training signals.
- Compare document-length tails with the context lengths intended for training.
  A long or short document is not automatically low quality.
- Inspect departures from the Zipf reference and rare token labels in context.
  The plots alone do not establish a universal cleaning rule.

Conclusions should follow the displayed run's evidence. Dataset-specific findings
from earlier runs are not carried over when `output_path` changes.


## Pre-processing and data curation

The configured processing chain normalizes text, removes empty documents,
removes exact duplicates, and applies selected MinHash deduplication. The cells
below inspect saved outputs. Standalone sweeps compare representations and
thresholds against shared inputs.

Training tokenization remains optional at the end. Perplexity filtering,
streaming, and additional processing stages are outside this minimal pipeline.


### Step 1 — Normalization

The ordered transformations, Unicode form, and newline limit come from
`pipeline_config['normalization']`, displayed in the setup cell. Policies can
preserve code whitespace or normalize prose depending on the saved settings.
Preparation removes empty documents after applying those transformations.


In [ ]:
cleaning_metrics = None
metrics_file = normalized_path / 'cleaning_metrics.csv'
if metrics_file.is_file():
    cleaning_metrics = pd.read_csv(metrics_file)
else:
    print('Cleaning metrics are missing. Run standalone preparation above.')


In [ ]:
if cleaning_metrics is not None:
    display(cleaning_metrics)
    for row in cleaning_metrics.itertuples(index=False):
        _LOGGER.info('%s: %d normalized documents; %d after empty removal (%.2f%% removed)',
                     row.split, row.after_normalization, row.after_empty_removal, row.removed_percentage)


### Step 2 — Exact deduplication

Exact deduplication hashes the configured text column after normalization.
Retained datasets, metrics, and duplicate pairs are saved separately. Pair
indices refer to the corresponding normalized input split before hashing.


In [ ]:
display(Markdown('**Standalone exact deduplication.** Reads the normalized stage and writes a separate destination.'))
show_command('run_exact_deduplication.py', '--input_path', normalized_path,
             '--output_path', exact_path)


In [ ]:
exact_datasets = {}
for split in configured_splits:
    path = exact_path / f'exact_dedup_{split}'
    if path.is_dir():
        exact_datasets[split] = load_from_disk(str(path))


In [ ]:
exact_dedup_metrics = None
metrics_file = exact_path / 'exact_dedup_metrics.csv'
if metrics_file.is_file():
    exact_dedup_metrics = pd.read_csv(metrics_file)
    display(exact_dedup_metrics)
else:
    print('Exact-deduplication metrics are missing. Run the command above.')


In [ ]:
preview_limit = 10
exact_duplicate_examples = None
pairs_file = exact_path / 'exact_duplicate_pairs.csv'
if pairs_file.is_file():
    exact_duplicate_examples = analysis.load_exact_duplicate_pairs(
        pairs_file, count=preview_limit, split='Training',
    )


In [ ]:
if exact_duplicate_examples is not None and 'train' in normalized_datasets:
    diagnostic.display_duplicate_pairs(
        normalized_datasets['train'],
        zip(exact_duplicate_examples['original_index'], exact_duplicate_examples['duplicate_index']),
        text_column=text_column, max_pairs=preview_limit,
    )


In [ ]:
if exact_dedup_metrics is not None and not exact_dedup_metrics.empty:
    fig, ax = diagnostic.plot_exact_deduplication(exact_dedup_metrics)
    plt.show()



<div style="max-width: 850px;">


### Step 3 - Near deduplication

---

Finding near duplicates help us to remove documents in the dataset that are not exact duplicates based on text, but instead have similar structure and content. We can define that two documents are near duplicates if they contain similar local patterns in their `n-grams` that are usually constructed in word level. 

N-grams have the following properties:

1. Preserve local structure,
2. Are robust to small details,
3. Ignore global ordering,
4. Are sensitive to similar/copied structure.

Using the N-grams of two documents, the similarity becomes comparing how much overlap have the two sets:

$$
J(A,B) = \frac{|A\cap B|}{|A\cup B|}.
$$

The above is the *Jaccard similarity*. However, at scale this becomes infeasible given that the complexity is $O(N^2)$, $N$ equal to the length of both sets. The solution is to use **MinHashing**. MinHashing converts the documents to a signature and allows us to compute the above expression with the property:

$$
P(\text{MinHash}(A)=\text{MinHash}(B)) = J(A,B).
$$

**The intuition** behind is that, for a permutation of an n-gram that is in $A\cup B$ it must be in: 1) in $A\cap B$, or 2) must be in A or B. The probability that lies in the intersection is $\frac{|A\cap B|}{|A\cup B|}$.

To implement **MinHashing** we simulate permutations with hash functions $h_i$ to create a document signature:

$$
m_i = h_i(g), g\in A.
$$

Then

$$
\text{signature}(A) = [m_1, m_2, m_3, ..., m_K],
$$

for $k=K$ hash functions. Then we compare two documents by computing:

$$
\hat{J}(A, B) = \frac{1}{K} \sum_{i=1}^K I[m_i(A)=m_i(B)].
$$

However, this computation is still $O(N^2)$. To speed up computation we turn our eyes to LSH (local sensitive hashing) to **help us to only compute the expensive comparison metric on documents that are similar in some bands**. Specifically, LSH performs the following steps:

1. Split each MinHash signature into $b$ bands, each containing $r$ rows:

$$
n = b\cdot r,
$$

where \(n\) is the signature length.

2. Hash each band independently into a hash table/bucket.

3. Define candidate pairs: two documents $A,B$ become a candidate pair if their signatures are identical in all $r$ rows of at least one band.

4 Only for these candidate pairs, compute a more precise similarity measure, such as $\hat{J}(A, B)$.


For example, for a two component band, a signature can become:
   
$$
\text{signature}(A) = [[m_1, m_2], [m_3, m_4], ..., [m_{K-1}, m_K]].
$$

Luckily, there are packages in Python that can help us computing these permutations - MinHashes and similarities with LSH. In this exercise we will use `datasketch`. In practice, we need to tune the following parameters:

- **`NGRAM size` — controls the representation.** We split each document into overlapping $n$-grams before computing MinHash. Smaller $n$-grams create more overlap between documents and are more tolerant of small textual changes, but can make unrelated documents appear similar. Larger $n$-grams capture more specific context and reduce accidental matches, but become more sensitive to small edits. **Trade-off:** smaller $n$ → higher recall / lower precision; larger $n$ → higher precision / lower recall.

- **`Number of permutations` — controls the accuracy of MinHash.** This determines the number of hash functions used to construct each MinHash signature. More permutations give a more accurate estimate of the true Jaccard similarity and reduce the variance of $\hat J(A,B)$, but increase hashing cost, memory usage, and LSH index size. **Trade-off:** more permutations → better similarity estimates but more compute and memory; fewer → faster and cheaper but noisier estimates.

- **`Similitud threshold` — controls which similarities LSH targets.** This tells LSH approximately where we want the transition between unlikely and likely candidate pairs to occur. A higher threshold retrieves fewer, more strongly similar candidates but risks missing useful moderately similar pairs. A lower threshold generates more candidates, improving recall but increasing false positives and downstream comparisons. **Trade-off:** higher threshold → higher precision / lower recall; lower threshold → higher recall / lower precision.

- **\(b\) and \(r\) — control the LSH banding behavior.** The MinHash signature is divided into \(b\) bands containing $r$ rows each, roughly satisfying $br=\text{NUM\_PERM}$. Two documents become candidates when they match on all \(r\) rows of at least one band. More bands with fewer rows make LSH more permissive; fewer bands with more rows make it stricter. **Trade-off:** more bands → higher recall / more false positives; more rows per band → higher precision / more false negatives.


</div>


### Compare saved n-gram experiments

Generate signatures separately for normalized and exact-deduplicated training
inputs: their row counts and fingerprints differ. Each signature set can be
shared by compatible trials against that same input. The commands below derive
the n-gram sizes and hash settings from the saved configuration, with enough
sizes for both the n-gram and threshold sweeps.

Each sweep writes to its own directory. Add `--save_dataset` for optional retained
dataset copies; the default saves indices and pairs without copying the input.


In [ ]:
display(Markdown('**Generate signatures before exact deduplication.**'))
show_command('run_minhash_signatures.py', '--input_path', normalized_path / 'normalized_train',
             '--output_path', normalized_signatures_path, '--ngram-sizes', *signature_ngram_sizes)


In [ ]:
display(Markdown('**Standalone n-gram sweep before exact deduplication.**'))
show_command('run_minhash_dedup_ngram_experiments.py',
             '--input_path', normalized_path / 'normalized_train',
             '--signatures_path', normalized_signatures_path,
             '--output_path', ngram_result_dirs['Before exact deduplication'])


In [ ]:
display(Markdown('**Generate signatures after exact deduplication.** Share this set with the threshold sweep below.'))
show_command('run_minhash_signatures.py', '--input_path', exact_path / 'exact_dedup_train',
             '--output_path', exact_signatures_path, '--ngram-sizes', *signature_ngram_sizes)


In [ ]:
display(Markdown('**Standalone n-gram sweep after exact deduplication.**'))
show_command('run_minhash_dedup_ngram_experiments.py',
             '--input_path', exact_path / 'exact_dedup_train',
             '--signatures_path', exact_signatures_path,
             '--output_path', ngram_result_dirs['After exact deduplication'])


Candidate totals and removal counts come directly from the selected runs'
summary CSVs. Missing optional experiments are skipped. Plot captions use saved
result settings, including any parameter overrides used for those trials.


In [ ]:
available_ngram_dirs = {
    label: directory for label, directory in ngram_result_dirs.items()
    if (directory / 'ngram_results.csv').is_file()
}
ngram_results_table = analysis.load_ngram_comparison(available_ngram_dirs) if available_ngram_dirs else None
if ngram_results_table is not None:
    display(ngram_results_table[[
        'dataset', 'ngram_size', 'total_documents', 'candidate_total',
        'removed_count', 'removed_percentage',
    ]])
else:
    print('No optional n-gram results found. Run the commands above to generate them.')


In [ ]:
if (ngram_results_table is not None and not ngram_results_table.empty
        and exact_dedup_metrics is not None):
    exact_training_removed = int(exact_dedup_metrics.set_index('split').loc['Training', 'removed_count'])
    fig, axes = diagnostic.plot_ngram_comparison(
        ngram_results_table, labels=available_ngram_dirs, exact_training_removed=exact_training_removed,
    )
    plt.show()


In [ ]:
for label, directory in available_ngram_dirs.items():
    saved_results = analysis.load_experiment_results(directory, kind='ngram')
    if saved_results is not None and not saved_results.empty:
        preview_ngram = int(saved_results.iloc[0]['ngram_size'])
        print(f'{label}, {preview_ngram}-gram: removed document preview')
        diagnostic.display_document_table(analysis.load_removed_documents(
            directory, preview_ngram, count=preview_limit,
        ))


#### Vary the similarity threshold

This independent sweep uses `threshold_experiment` from the saved configuration.
With `--signatures_path`, it reuses the compatible after-exact signature set.
Omitting that argument computes signatures in memory. All metrics and pairs
refer to the exact saved input named in each trial's metadata.


In [ ]:
display(Markdown('**Standalone threshold sweep.** Writes a separate summary and trial directories.'))
show_command('run_minhash_dedup_threshold_experiments.py',
             '--input_path', exact_path / 'exact_dedup_train',
             '--signatures_path', exact_signatures_path, '--output_path', threshold_results_dir)


In [ ]:
threshold_results_table = analysis.load_experiment_results(threshold_results_dir, kind='threshold')
if threshold_results_table is not None:
    display(threshold_results_table)


In [ ]:
if threshold_results_table is not None and not threshold_results_table.empty:
    fig, ax = diagnostic.plot_threshold_results(threshold_results_table)
    plt.show()


In [ ]:
inspect_threshold = (
    float(threshold_results_table.iloc[0]['similarity_threshold'])
    if threshold_results_table is not None and not threshold_results_table.empty
    else float(pipeline_config['threshold_experiment']['similarity_thresholds'][0])
)
display(Markdown('**Run one threshold independently.** `--thresholds` overrides the configured list. '
                 'This uses a separate destination so the full sweep summary is preserved. '
                 'Add `--overwrite` to rerun that destination, or `--save_dataset` to export retained rows.'))
show_command('run_minhash_dedup_threshold_experiments.py',
             '--input_path', exact_path / 'exact_dedup_train', '--signatures_path', exact_signatures_path,
             '--output_path', single_threshold_dir, '--thresholds', inspect_threshold)


In [ ]:
matched_pairs = None
pair_metadata = None
inspection_results_dir = (
    single_threshold_dir if (single_threshold_dir / 'threshold_results.csv').is_file()
    else threshold_results_dir
)
inspection_results = analysis.load_experiment_results(inspection_results_dir, kind='threshold')
if inspection_results is not None and not inspection_results.empty:
    experiment_id = str(inspection_results.iloc[0]['experiment_id'])
    pair_result = analysis.load_duplicate_pairs(inspection_results_dir, experiment_id)
    if pair_result is not None:
        matched_pairs, pair_metadata = pair_result
        inspect_threshold = float(pair_metadata['similarity_threshold'])
        pair_source_dataset = analysis.load_source_dataset(pair_metadata)
        diagnostic.display_similarity_extremes(
            pair_source_dataset, matched_pairs, count=preview_limit,
            text_column=pair_metadata['text_column'],
        )


In [ ]:
if matched_pairs is not None:
    fig, ax = diagnostic.plot_matched_similarity_distribution(matched_pairs, inspect_threshold)
    if fig is not None:
        plt.show()


In [ ]:
display(Markdown('**Standalone selected MinHash deduplication.** Uses `minhash_deduplication` from the saved config.'))
show_command('run_minhash_deduplication.py', '--input_path', exact_path,
             '--output_path', minhash_path)


In [ ]:
selected_datasets = {}
for split in configured_splits:
    path = minhash_path / f'minhash_dedup_{split}'
    if path.is_dir():
        selected_datasets[split] = load_from_disk(str(path))


In [ ]:
metrics_file = minhash_path / 'minhash_dedup_metrics.csv'
if metrics_file.is_file():
    minhash_dedup_metrics = pd.read_csv(metrics_file)
    display(minhash_dedup_metrics)
    for row in minhash_dedup_metrics.itertuples(index=False):
        _LOGGER.info('%s: %d -> %d documents (%.2f%% removed by MinHash)',
                     row.split, row.before, row.after, row.removed_percentage)
else:
    print('Selected MinHash metrics are missing. Run the command above.')


<div style="max-width: 850px;">

### Step 4 - Tokenization

---

A dataset consist of documents. However, dataset is usually regarded as amount of tokens it cointains. Even more, when training a model we usually refer to the total count of tokens the model saw during training considering epochs too - e.g. GPT4 was trained on XX tokens. GPT models use a **Byte-Pair Encoding (BPE)** tokenizers, which decompose words into sub-words based on the most frequent pair of characters in subwords.

As an example, consider the input dataset `'low', 'lower', 'lowest'`. BPE works as follows

1. **Initialize vocabulary** - start with all unique characters as tokens including a token for unknown:

Vocabulary: `'l', 'o', 'w', 'e', 'r', 's', 't', '<unk>'`

2. **Count token pairs in the dataset**

```
(l,o) = 3    (w,e) = 2   (e,s) = 1
(o,w) = 3   (e,r) = 2   (s,t) = 1
```

3. **Merge most frequent pair into new subword token**:

Vocabulary after merging `(l,o)`: `'lo', 'w', 'e', 'r', 's', 't', '<unk>'`

4. **Repeat the process until desired granularity**:

Final vocabulary : `'low', 'lower', 'est', '<unk>'`

5. **The dictionary of vocabulary comprise of**:

`{'low':0, 'lower':1, 'est':3, '<unk>':3}`


The following optional cells use the tokenizer recorded in the selected run. They process only the configured splits. Leave `prepare_training_data = False` for read-only analysis; set it to `True` only to tokenize and export training inputs. A locally available tokenizer or Hub access is required for that optional step.

</div>


In [ ]:
# Explicit opt-in: the default notebook run only reads saved artifacts.
prepare_training_data = False
tokenized_datasets = {}
if prepare_training_data:
    from transformers import AutoTokenizer

    training_tokenizer = AutoTokenizer.from_pretrained(
        pipeline_config['tokenizer']['name_or_path'],
        revision=pipeline_config['tokenizer'].get('revision'),
    )

    def preprocess_tokenizer_update(batch):
        return {'tokenized': training_tokenizer(batch[text_column])['input_ids']}

    for split, dataset in selected_datasets.items():
        tokenized_datasets[split] = dataset.map(
            preprocess_tokenizer_update, batched=True,
            batch_size=pipeline_config['batch_size'], num_proc=pipeline_config['num_proc'],
            desc=f'Tokenizing {split}',
        )


## Optional training export

When opted in above, save tokenized splits beneath the selected run's `tokenized/`
directory. Existing exports are refused. The analysis workflow itself ends with
the saved metrics and previews; scoring and model-training examples are separate
from this pipeline.


In [ ]:
if prepare_training_data:
    training_output = output_path / 'tokenized'
    destinations = {split: training_output / f'tokenized_{split}' for split in tokenized_datasets}
    if any(path.exists() for path in destinations.values()):
        raise FileExistsError('Tokenized exports already exist; choose a new export destination.')
    for split, dataset in tokenized_datasets.items():
        dataset.save_to_disk(str(destinations[split]))
